# Run ResNet-18 (and the full pipeline) on a free GPU in Google Colab

ResNet-18 at 224 x 224 is very slow on a laptop CPU. In Colab: **Runtime > Change runtime type > T4 GPU**, then run the cells in order.

Results are copied to your Google Drive at the end, so download them into your laptop's `outputs/` folder.

In [ ]:
# 1. Check the GPU is on (should print a GPU name, e.g. Tesla T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code
The repository is private, so git asks for a token. The token is typed into a hidden box and is not saved in the notebook.

In [ ]:
import getpass, subprocess, os
token = getpass.getpass('GitHub token (repo scope): ')
url = f'https://{token}@github.com/sampathvajjiparthi-a11y/fic-cfx.git'
subprocess.run(['git', 'clone', '-q', url, '/content/fic-cfx'], check=True)
del token, url
os.chdir('/content/fic-cfx')
print(os.listdir('.'))

In [ ]:
# 3. Install the project (PyTorch is already installed in Colab)
!pip install -q -e . statsmodels

## 4. Big dataset (optional)
On your laptop, run `00_prepare_fashion_products.py`, zip the folder `data/raw/fashion_products` and upload the zip to **My Drive/fic-cfx/fashion_products.zip**. Skip this cell if you only run Fashion-MNIST.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
zip_path = '/content/drive/MyDrive/fic-cfx/fashion_products.zip'
if os.path.exists(zip_path):
    !mkdir -p data/raw && unzip -q -o "$zip_path" -d data/raw/
    print(sorted(os.listdir('data/raw/fashion_products')))
else:
    print('No big dataset zip found, only Fashion-MNIST will be used.')

## 5. Quick check, then train
The quick run takes a few minutes and only checks that everything works.

In [ ]:
!python scripts/run_pipeline.py --dataset small --quick

In [ ]:
# Fine-tune ResNet-18 on both datasets (5 seeds each, from configs/default.yaml)
!python scripts/03_train_deep.py --dataset small --model resnet18
!python scripts/03_train_deep.py --dataset big --model resnet18

In [ ]:
# Or run EVERYTHING for both datasets (all 4 models, XAI, counterfactuals, statistics)
# !python scripts/run_pipeline.py --dataset small
# !python scripts/run_pipeline.py --dataset big

## 6. Save results to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/fic-cfx
!zip -q -r /content/drive/MyDrive/fic-cfx/outputs_colab.zip outputs
print('Saved to My Drive/fic-cfx/outputs_colab.zip')